In [ ]:
import os
import csv
import re
import json
import random
import requests
import unicodedata
import logging
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import time

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cấu hình
BASE_URL = "https://mixkit.co"
CATEGORIES = [
    "food", "drink", "coffee", "tea", "fruit", "corn", "beer",
    "cocktail", "salad", "eating", "vegetable", "fast-food", "restaurant"
]
MAX_PAGES = 100
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko) Version/17.0 Mobile/15E148 Safari/537.36"
]
OUTPUT_DIR = r"D:\project\Food-Captioning\mixkit"
IMAGE_FOLDER = os.path.join(OUTPUT_DIR, "image")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "data.csv")
NUM_WORKERS_CRAWL = 4
NUM_WORKERS_DOWNLOAD = min(os.cpu_count() * 2, 16)
TIMEOUT = 15
CHUNK_SIZE = 16384
MAX_RETRIES = 3

# Tạo thư mục
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGE_FOLDER, exist_ok=True)

# Tạo session với retry
session = requests.Session()
session.headers.update({"Accept-Language": "en-US,en;q=0.9"})
retries = Retry(total=MAX_RETRIES, backoff_factor=0.1, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

def get_headers():
    """Chọn ngẫu nhiên User-Agent"""
    return {"User-Agent": random.choice(USER_AGENTS), "Referer": "https://google.com"}

def clean_string(text):
    """Làm sạch chuỗi để tránh lỗi mã hóa"""
    if not isinstance(text, str):
        return str(text)
    return unicodedata.normalize('NFC', text).encode('utf-8', 'ignore').decode('utf-8')

def get_filename_from_url(url):
    """Trích xuất tên file từ URL với tiền tố mixkit_"""
    if not url:
        return None
    filename = url.split('/')[-1].split('?')[0]
    filename = re.sub(r'[^a-zA-Z0-9\-\.]', '_', filename)
    if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        filename += '.jpg'
    return f"mixkit_{filename}"

def download_image(video, csv_writer):
    """Tải ảnh và ghi vào CSV"""
    url = video.get('Thumbnail', '')
    title = clean_string(video.get('Title', ''))
    description = clean_string(video.get('Description', ''))
    if not url:
        csv_writer.writerow({
            'id': '', 'Thumbnail': url, 'Title': title, 'Description': description, 'status': 'empty_url'
        })
        return logger.info("❌ URL trống")

    filename = get_filename_from_url(url)
    if not filename:
        csv_writer.writerow({
            'id': '', 'Thumbnail': url, 'Title': title, 'Description': description, 'status': 'invalid_url'
        })
        return logger.info(f"❌ URL không hợp lệ: {url}")

    file_path = os.path.join(IMAGE_FOLDER, filename)
    base, ext = os.path.splitext(filename)
    counter = 1
    while os.path.exists(file_path):
        filename = f"{base}_{counter}{ext}"
        file_path = os.path.join(IMAGE_FOLDER, filename)
        counter += 1

    if os.path.exists(file_path):
        csv_writer.writerow({
            'id': filename, 'Thumbnail': url, 'Title': title, 'Description': description, 'status': 'already_exists'
        })
        return logger.info(f"✅ Đã có: {filename}")

    try:
        with session.get(url, timeout=TIMEOUT, stream=True) as r:
            r.raise_for_status()
            with open(file_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
        csv_writer.writerow({
            'id': filename, 'Thumbnail': url, 'Title': title, 'Description': description, 'status': 'success'
        })
        return logger.info(f"📥 OK: {filename}")
    except requests.RequestException as e:
        csv_writer.writerow({
            'id': filename, 'Thumbnail': url, 'Title': title, 'Description': description, 'status': f'failed_{str(e).lower()}'
        })
        return logger.error(f"❌ Lỗi {filename}: {e}")

def scrape_category(category, seen_thumbnails, csv_writer):
    """Thu thập dữ liệu và tải ảnh từ một danh mục"""
    videos = []
    category_url = f"{BASE_URL}/free-stock-video/{category}/"
    logger.info(f"\n📂 Đang thu thập danh mục: {category}")

    for page in range(1, MAX_PAGES + 1):
        page_url = f"{category_url}?page={page}"
        logger.info(f"🔍 Trang {page}: {page_url}")

        try:
            response = session.get(page_url, headers=get_headers(), timeout=TIMEOUT)
            if response.status_code != 200:
                logger.warning(f"❌ HTTP {response.status_code}: Dừng tại trang {page}")
                break

            soup = BeautifulSoup(response.text, "html.parser")
            video_items = soup.select(".item-grid-card")
            if not video_items:
                logger.info(f"✅ Hết dữ liệu tại trang {page}")
                break

            page_videos = []
            for item in video_items:
                title = item.select_one(".item-grid-card__title a")
                description = item.select_one(".item-grid-card__description")
                thumbnail = item.select_one(".item-grid-video-player__thumb")
                video_link = item.select_one(".item-grid-video-player__video")

                video = {
                    "Title": title.text.strip() if title else "Không có tiêu đề",
                    "Description": description.text.strip() if description else "Không có mô tả",
                    "Thumbnail": urljoin(BASE_URL, thumbnail["src"]) if thumbnail else "",
                    "Video": urljoin(BASE_URL, video_link["src"]) if video_link else ""
                }

                # Kiểm tra trùng lặp Thumbnail
                if video['Thumbnail'] and video['Thumbnail'] not in seen_thumbnails:
                    seen_thumbnails.add(video['Thumbnail'])
                    page_videos.append(video)

            # Tải ảnh song song cho trang
            with ThreadPoolExecutor(max_workers=NUM_WORKERS_DOWNLOAD) as executor:
                futures = [executor.submit(download_image, video, csv_writer) for video in page_videos]
                for future in as_completed(futures):
                    future.result()

            videos.extend(page_videos)

        except requests.RequestException as e:
            logger.error(f"⚠️ Lỗi tại {page_url}: {e}")
            break

        time.sleep(random.uniform(1, 3))

    # Lưu JSON tạm thời cho danh mục
    save_path = os.path.join(OUTPUT_DIR, f"mixkit_{category}.json")
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(videos, f, indent=4, ensure_ascii=False)
    logger.info(f"✅ Lưu {len(videos)} video vào {save_path}")

    return videos

"""Chạy toàn bộ quy trình"""
logger.info("🚀 Bắt đầu thu thập và tải ảnh...")

# Tạo CSV
with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'Thumbnail', 'Title', 'Description', 'status'])
    writer.writeheader()

# Theo dõi Thumbnail đã thấy
seen_thumbnails = set()
all_videos = []

# Thu thập và tải song song các danh mục
with open(OUTPUT_CSV, mode='a', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'Thumbnail', 'Title', 'Description', 'status'])
    with ThreadPoolExecutor(max_workers=NUM_WORKERS_CRAWL) as executor:
        futures = {executor.submit(scrape_category, cat, seen_thumbnails, writer): cat for cat in CATEGORIES}
        for future in as_completed(futures):
            videos = future.result()
            all_videos.extend(videos)

logger.info(f"\n📊 Tổng số video thu thập: {len(all_videos)}")
logger.info(f"📊 Tổng số Thumbnail duy nhất: {len(seen_thumbnails)}")
logger.info(f"✅ Hoàn tất! Kết quả lưu tại: {OUTPUT_CSV}")

2025-04-24 21:07:50,863 - INFO - 🚀 Bắt đầu thu thập và tải ảnh...
2025-04-24 21:07:50,898 - INFO - 
📂 Đang thu thập danh mục: food
2025-04-24 21:07:50,912 - INFO - 🔍 Trang 1: https://mixkit.co/free-stock-video/food/?page=1
2025-04-24 21:07:50,918 - INFO - 
📂 Đang thu thập danh mục: drink
2025-04-24 21:07:50,930 - INFO - 🔍 Trang 1: https://mixkit.co/free-stock-video/drink/?page=1
2025-04-24 21:07:50,942 - INFO - 
📂 Đang thu thập danh mục: coffee
2025-04-24 21:07:50,963 - INFO - 🔍 Trang 1: https://mixkit.co/free-stock-video/coffee/?page=1
2025-04-24 21:07:50,967 - INFO - 
📂 Đang thu thập danh mục: tea
2025-04-24 21:07:50,988 - INFO - 🔍 Trang 1: https://mixkit.co/free-stock-video/tea/?page=1
2025-04-24 21:07:54,360 - INFO - 📥 OK: mixkit_4801-thumb-360-0_1.jpg
2025-04-24 21:07:54,402 - INFO - 📥 OK: mixkit_236-thumb-360-0_1.jpg
2025-04-24 21:07:54,423 - INFO - 📥 OK: mixkit_4919-thumb-360-0_1.jpg
2025-04-24 21:07:54,423 - INFO - 📥 OK: mixkit_4989-thumb-360-0_1.jpg
2025-04-24 21:07:54,424 - I

In [ ]:
import os
import csv
import shutil
import pandas as pd
import numpy as np
import tensorflow as tf
import logging
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.mixed_precision import Policy, set_global_policy

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Cấu hình
CONFIG = {
    'input_csv': r"D:\project\Food-Captioning\mixkit\data.csv",
    'image_folder': r"D:\project\Food-Captioning\mixkit\images",
    'model_path': r"D:\project\Food-Captioning\model\best_resnet_model.h5",
    'output_food': r"D:\project\Food-Captioning\mixkit\sorted\food",
    'output_nonfood': r"D:\project\Food-Captioning\mixkit\sorted\nonfood",
    'output_class_csv': r"D:\project\Food-Captioning\mixkit\data_with_class.csv",
    'batch_size': 64,
    'valid_extensions': ('.jpg', '.jpeg', '.png')
}

# Tạo thư mục
os.makedirs(CONFIG['output_food'], exist_ok=True)
os.makedirs(CONFIG['output_nonfood'], exist_ok=True)

# Bật mixed precision để tăng tốc và giảm bộ nhớ
set_global_policy(Policy('mixed_float16'))

def clean_string(text):
    """Làm sạch chuỗi để tránh lỗi mã hóa"""
    if not isinstance(text, str):
        return str(text)
    return unicodedata.normalize('NFC', text).encode('utf-8', 'ignore').decode('utf-8')

def predict_batch(img_paths, model, batch_size):
    """Dự đoán phân loại cho một lô ảnh"""
    batch_imgs = []
    valid_paths = []
    for img_path in img_paths:
        if os.path.exists(img_path) and img_path.lower().endswith(CONFIG['valid_extensions']):
            try:
                img = image.load_img(img_path, target_size=(224, 224))
                img_array = image.img_to_array(img)
                batch_imgs.append(img_array)
                valid_paths.append(img_path)
            except Exception as e:
                logger.error(f"❌ Lỗi tải ảnh {img_path}: {e}")
    if not batch_imgs:
        return [], [], []

    batch_imgs = np.array(batch_imgs)
    batch_imgs = preprocess_input(batch_imgs)
    predictions = model.predict(batch_imgs, batch_size=batch_size, verbose=0)
    labels = [0 if pred < 0.5 else 1 for pred in predictions.flatten()]
    confidences = [float(1 - pred) if label == 0 else float(pred) for label, pred in zip(labels, predictions.flatten())]
    return valid_paths, labels, confidences

def classify_images():
    """Phân loại ảnh thành Food/Non-food"""
    logger.info("🚀 Bắt đầu phân loại ảnh...")

    # Tải mô hình
    try:
        model = load_model(CONFIG['model_path'])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    except Exception as e:
        logger.error(f"❌ Lỗi tải mô hình {CONFIG['model_path']}: {e}")
        return

    # Đọc data.csv
    try:
        df = pd.read_csv(CONFIG['input_csv'], encoding='utf-8-sig')
    except UnicodeDecodeError:
        logger.warning("⚠️ Lỗi mã hóa UTF-8, thử latin1...")
        df = pd.read_csv(CONFIG['input_csv'], encoding='latin1')
    except Exception as e:
        logger.error(f"❌ Lỗi khi đọc {CONFIG['input_csv']}: {e}")
        return

    # Làm sạch chuỗi
    df['Title'] = df['Title'].apply(clean_string)
    df['Description'] = df['Description'].apply(clean_string)

    # Thêm cột class và confidence
    df['class'] = -1
    df['confidence'] = 0.0

    # Kiểm tra file tồn tại
    df['file_exists'] = df['id'].apply(lambda x: os.path.exists(os.path.join(CONFIG['image_folder'], x)) and x.lower().endswith(CONFIG['valid_extensions']))
    missing_files = df[~df['file_exists']]['id'].tolist()
    if missing_files:
        logger.warning(f"⚠️ Thiếu {len(missing_files)} file: {missing_files[:5]}...")
    df = df[df['file_exists']].copy()
    df.drop(columns=['file_exists'], inplace=True)
    logger.info(f"📊 Số ảnh hợp lệ: {len(df)}")

    # Phân loại theo lô
    results = []
    img_paths = [os.path.join(CONFIG['image_folder'], row['id']) for _, row in df.iterrows()]
    for i in range(0, len(img_paths), CONFIG['batch_size']):
        batch_paths = img_paths[i:i + CONFIG['batch_size']]
        valid_paths, labels, confidences = predict_batch(batch_paths, model, CONFIG['batch_size'])
        for img_path, label, confidence in zip(valid_paths, labels, confidences):
            img_id = os.path.basename(img_path)
            try:
                dest_dir = CONFIG['output_food'] if label == 0 else CONFIG['output_nonfood']
                dest_path = os.path.join(dest_dir, img_id)
                base, ext = os.path.splitext(dest_path)
                counter = 1
                while os.path.exists(dest_path):
                    dest_path = f"{base}_{counter}{ext}"
                    counter += 1
                shutil.copy(img_path, dest_path)
                results.append((img_id, label, confidence))
                logger.info(f"📸 {img_id} → {'Food' if label == 0 else 'Non-food'} ({confidence*100:.2f}%)")
            except Exception as e:
                logger.error(f"❌ Lỗi sao chép {img_path}: {e}")
                results.append((img_id, -1, 0.0))
        # Ghi log cho các file không hợp lệ
        for img_path in batch_paths:
            if img_path not in valid_paths:
                img_id = os.path.basename(img_path)
                results.append((img_id, -1, 0.0))
                logger.error(f"❌ File không hợp lệ: {img_path}")

    # Cập nhật DataFrame
    for img_id, label, confidence in results:
        if label != -1:
            df.loc[df['id'] == img_id, 'class'] = label
            df.loc[df['id'] == img_id, 'confidence'] = confidence

    # Lưu kết quả
    df.to_csv(CONFIG['output_class_csv'], index=False, encoding='utf-8-sig')
    logger.info(f"✅ Phân loại hoàn tất! Kết quả lưu tại: {CONFIG['output_class_csv']}")

classify_images()

2025-04-24 21:23:36,333 - INFO - 🚀 Bắt đầu phân loại ảnh...
c:\Users\ASUS\anaconda3\envs\python10\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
2025-04-24 21:23:40,350 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
2025-04-24 21:23:40,379 - WARNING - Error in loading the saved optimizer state. As a result, your model is starting with a freshly initialized optimizer.
2025-04-24 21:23:40,751 - WARNING - ⚠️ Thiếu 1 file: ['Bmixkit_4930-thumb-360-0.jpg']...
2025-04-24 21:23:40,758 - INFO - 📊 Số ảnh hợp lệ: 2200
2025-04-24 21:23:54,315 - INFO - 📸 mixkit_4801-thumb-360-0_1.jpg → Non-food (99.99%)
2025-04-24 21:23:54,319 - INFO - 📸 mixkit_236-thumb-360-0_1.jpg → Food (100.00%)
2025-04-24 21:23:54,321 - INFO - 📸 mixkit_4919-thumb-360-0_1.jpg → Food (75.30%)
2025-04-24 21:23:5